# Fast and Safe - Validación del DW
Verifica que todas las tablas quedaron bien cargadas y responde las preguntas del enunciado.

In [ ]:
import pandas as pd
import yaml
from sqlalchemy import create_engine

with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)

cfg = config['TARGET_DB']
url = f"{cfg['drivername']}://{cfg['user']}:{cfg['password']}@{cfg['host']}:{cfg['port']}/{cfg['dbname']}"
dw = create_engine(url)
print('Conexión al DW OK')

## 1. Conteo de filas por tabla

In [ ]:
tablas = [
    'dim_fecha', 'dim_cliente', 'dim_sede', 'dim_mensajero',
    'dim_ciudad', 'dim_tipo_servicio', 'dim_estado_servicio',
    'dim_tipo_novedad', 'hecho_servicio', 'hecho_transiciones_estado',
    'resumen_novedades'
]
for t in tablas:
    try:
        n = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', dw).iloc[0,0]
        print(f'  {t:35s}: {n} filas')
    except Exception as e:
        print(f'  {t:35s}: ERROR - {e}')

## 2. Vista previa dimensiones

In [ ]:
pd.read_sql('SELECT * FROM dim_fecha LIMIT 5', dw)

In [ ]:
pd.read_sql('SELECT * FROM dim_cliente', dw)

In [ ]:
pd.read_sql('SELECT * FROM dim_mensajero', dw)

In [ ]:
pd.read_sql('SELECT * FROM dim_estado_servicio ORDER BY orden_estado', dw)

In [ ]:
pd.read_sql('SELECT * FROM dim_tipo_novedad', dw)

## 3. Preguntas del enunciado
### P1 - ¿En qué meses del año los clientes solicitan más servicios?

In [ ]:
pd.read_sql("""
    SELECT f.month, f.nombre_mes, COUNT(*) AS total_servicios
    FROM hecho_servicio h
    JOIN dim_fecha f ON f.key_dim_fecha = h.fk_fecha_solicitud
    GROUP BY f.month, f.nombre_mes
    ORDER BY total_servicios DESC
""", dw)

### P2 - ¿Cuáles son los días con más solicitudes?

In [ ]:
pd.read_sql("""
    SELECT f.weekday, f.nombre_dia, COUNT(*) AS total
    FROM hecho_servicio h
    JOIN dim_fecha f ON f.key_dim_fecha = h.fk_fecha_solicitud
    GROUP BY f.weekday, f.nombre_dia
    ORDER BY f.weekday
""", dw)

### P4 - Número de servicios por cliente y por mes

In [ ]:
pd.read_sql("""
    SELECT c.nombre_cliente, f.month, f.nombre_mes, COUNT(*) AS servicios
    FROM hecho_servicio h
    JOIN dim_cliente c ON c.key_dim_cliente = h.fk_cliente
    JOIN dim_fecha   f ON f.key_dim_fecha   = h.fk_fecha_solicitud
    GROUP BY c.nombre_cliente, f.month, f.nombre_mes
    ORDER BY c.nombre_cliente, f.month
""", dw)

### P5 - Mensajeros más eficientes

In [ ]:
pd.read_sql("""
    SELECT m.nombre, COUNT(*) AS servicios_prestados,
           ROUND(AVG(h.tiempo_total_minutos), 1) AS tiempo_promedio_min
    FROM hecho_servicio h
    JOIN dim_mensajero m ON m.key_dim_mensajero = h.fk_mensajero
    GROUP BY m.nombre
    ORDER BY servicios_prestados DESC
""", dw)

### P6 - Sedes que más servicios solicitan por cliente

In [ ]:
pd.read_sql("""
    SELECT c.nombre_cliente, s.nombre_sede, s.ciudad, COUNT(*) AS total_servicios
    FROM hecho_servicio h
    JOIN dim_cliente c ON c.key_dim_cliente = h.fk_cliente
    JOIN dim_sede    s ON s.key_dim_sede    = h.fk_sede
    GROUP BY c.nombre_cliente, s.nombre_sede, s.ciudad
    ORDER BY c.nombre_cliente, total_servicios DESC
""", dw)

### P7 - Tiempo promedio de entrega

In [ ]:
pd.read_sql("""
    SELECT
        ROUND(AVG(tiempo_total_minutos), 2)       AS promedio_total_min,
        ROUND(AVG(tiempo_fase_asignacion_min), 2) AS promedio_asignacion_min,
        ROUND(AVG(tiempo_fase_recogida_min), 2)   AS promedio_recogida_min,
        ROUND(AVG(tiempo_fase_entrega_min), 2)    AS promedio_entrega_min
    FROM hecho_servicio
""", dw)

### P8 - ¿En qué fase del servicio hay más demoras?

In [ ]:
pd.read_sql("""
    SELECT
        e.nombre_estado,
        e.orden_estado,
        ROUND(AVG(t.duracion_en_estado_min), 2) AS duracion_promedio_min,
        COUNT(*) AS num_transiciones
    FROM hecho_transiciones_estado t
    JOIN dim_estado_servicio e ON e.key_dim_estado = t.fk_estado_servicio
    GROUP BY e.nombre_estado, e.orden_estado
    ORDER BY e.orden_estado
""", dw)

### P9 - ¿Cuáles son las novedades que más se presentan?

In [ ]:
pd.read_sql("""
    SELECT nombre_tipo_novedad, total_ocurrencias, servicios_afectados
    FROM resumen_novedades
    ORDER BY total_ocurrencias DESC
""", dw)